# Parte III — Evidencia de contenido y revisión de recomendaciones

Diagnóstico limitado al entrenamiento anterior a 2023 y a los resultados de validación de 2023. No se consultan resultados de la prueba de 2024 para diseñar el enfoque por contenido. La prueba 2024 ya fue examinada previamente para otros modelos: cualquier evaluación adicional en esa ventana sería exploratoria, no una prueba final independiente. El corte por adjudicación no acredita disponibilidad histórica de las versiones actuales.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'docs').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.features.build_part1 import directory_fingerprint
base=ROOT/'data/processed'; inputs=['fact_award_supplier','fact_award','fact_award_item','recommendation']
for name in inputs:
    p=base/name
    assert (p/'_SUCCESS').is_file(), f'Entrada incompleta: {p}'
    files=list(p.rglob('*.parquet')); size=sum(f.stat().st_size for f in files)
    rows=sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    assert rows <= 500_000 and size <= 256*1024**2, f'Entrada excede límite: {p}'
    print(name, 'filas=', rows, 'bytes=', size, 'huella=', directory_fingerprint(p))
recmeta=json.loads((base/'recommendation/_metadata.json').read_text(encoding='utf-8'))
assert recmeta['input_sha256']==directory_fingerprint(base/'fact_award_supplier'), 'Cambiaron eventos'
for name in ['recommendation_baseline','recommendation_neighbors','recommendation_als']:
    path=ROOT/'artifacts'/name
    assert (path/'_SUCCESS').is_file()
    meta=json.loads((path/'_metadata.json').read_text(encoding='utf-8'))
    assert meta['input_sha256']==directory_fingerprint(base/'recommendation')
def read(name, columns):
    return ds.dataset(base/name,format='parquet',partitioning='hive',exclude_invalid_files=True).to_table(columns=columns).to_pandas()
def nonempty(s): return s.notna() & s.fillna('').astype(str).str.strip().ne('')

fact_award_supplier filas= 128395 bytes= 10754866 huella= 92dedb5c56ac4a52399b3df3f4283a51731c7931d578ca7ff57b64e7e3611734
fact_award filas= 128395 bytes= 23352125 huella= 714151596251c2e09573d393d3a5a747d604ede5134db8603242cb83f62b2778


fact_award_item filas= 137292 bytes= 37228336 huella= 850f894cef270ab7b256a54ddd523393079201d3149febed498c76ea9bc52e51
recommendation filas= 260581 bytes= 3276163 huella= 64ec49417ae4a8b0729709e358b4860be3c88574da6f9d273b0e65764cee135b


## Cobertura y grano de los campos

Cada evento elegible corresponde a adjudicación × proveedor. El texto de la licitación se repite por proveedor; el texto de ítem se conserva por adjudicación × proveedor × ítem. La presencia de texto o de un código no garantiza que sea útil ni que corresponda a información conocida históricamente.

In [2]:
key=['process_ocid','award_id','supplier_id']
s=read('fact_award_supplier',key+['entity_id','supplier_name','award_date','award_year','procurement_category'])
a=read('fact_award',['process_ocid','award_id','entity_name','tender_title','tender_description'])
i=read('fact_award_item',key+['item_id','item_description','item_classification_id'])
eligible=nonempty(s.entity_id)&nonempty(s.supplier_id)&s.award_date.notna()&(s.award_year<2023)
s=s.loc[eligible].copy()
assert not s.duplicated(key).any()
j=s.merge(a,on=['process_ocid','award_id'],validate='many_to_one')
items=i.merge(s[key+['entity_id']],on=key,validate='many_to_one')
rows=[]
for frame, name, field in [(j,'evento','procurement_category'),(j,'evento','tender_title'),(j,'evento','tender_description'),
                           (items,'ítem-proveedor','item_description'),(items,'ítem-proveedor','item_classification_id')]:
    valid=nonempty(frame[field]); values=frame.loc[valid,field].astype(str).str.strip()
    rows.append({'grano':name,'campo':field,'filas':len(frame),'no vacías':int(valid.sum()),
                 'cobertura':valid.mean(),'entidades':frame.loc[valid,'entity_id'].nunique(),
                 'proveedores':frame.loc[valid,'supplier_id'].nunique(),
                 'valores distintos':values.nunique(),'longitud < 10':int(values.str.len().lt(10).sum())})
display(pd.DataFrame(rows).set_index('campo').round(4))
print('Categorías por evento:'); display(j.procurement_category.value_counts(dropna=False))
print('La presencia de código no lo valida ni acredita que dos ítems sean equivalentes.')

,grano,filas,no vacías,cobertura,entidades,proveedores,valores distintos,longitud < 10
campo,,,,,,,,
procurement_category,evento,46636,46636,1.0000,2691,28878,3,46636
tender_title,evento,46636,46636,1.0000,2691,28878,41286,0
tender_description,evento,46636,46636,1.0000,2691,28878,42512,17
item_description,ítem-proveedor,50647,50647,1.0000,2691,28878,46642,861
item_classification_id,ítem-proveedor,50647,46357,0.9153,2669,27283,13428,8


Categorías por evento:


procurement_category
goods       21148
services    19383
works        6105
Name: count, dtype: int64

La presencia de código no lo valida ni acredita que dos ítems sean equivalentes.


## Comparación de casos de validación

Se elige determinísticamente una entidad con acierto de ALS, otra sin acierto y una sin ranking de vecindario. Se muestra solo el top 3. Las categorías son antecedentes del proveedor y la entidad en entrenamiento; no prueban equivalencia de productos. ALS no ofrece una explicación causal por recomendación.

In [3]:
path=ROOT/'artifacts'
als_metrics=pd.read_parquet(path/'recommendation_als/metrics_by_entity.parquet')
als_metrics=als_metrics[(als_metrics['rank']==json.loads((path/'recommendation_als/_metadata.json').read_text(encoding='utf-8'))['selected']['rank'])&(als_metrics.k==10)].sort_values('entity_id')
nb_metrics=pd.read_parquet(path/'recommendation_neighbors/metrics_by_entity.parquet')
nb_metrics=nb_metrics[nb_metrics.k==10].sort_values('entity_id')
choices=[('ALS con acierto',als_metrics.loc[als_metrics.hits.gt(0),'entity_id'].iloc[0]),
         ('ALS sin acierto',als_metrics.loc[als_metrics.hits.eq(0),'entity_id'].iloc[0]),
         ('Vecindario vacío',nb_metrics.loc[nb_metrics.ranking_length.eq(0) & ~nb_metrics.entity_id.isin([als_metrics.loc[als_metrics.hits.gt(0),'entity_id'].iloc[0], als_metrics.loc[als_metrics.hits.eq(0),'entity_id'].iloc[0]]),'entity_id'].iloc[0])]
future=ds.dataset(base/'recommendation/2023/future_pairs',format='parquet',exclude_invalid_files=True).to_table(columns=['entity_id','supplier_id','eligible_positive']).to_pandas()
rankings={m:pd.read_parquet(path/f'recommendation_{folder}/rankings.parquet') for m,folder in [('popularidad','baseline'),('vecindario','neighbors'),('ALS','als')]}
cat_entity=j.groupby('entity_id').procurement_category.agg(lambda x: ', '.join(sorted(set(x.dropna())))).to_dict()
cat_supplier=j.groupby('supplier_id').procurement_category.agg(lambda x: ', '.join(sorted(set(x.dropna())))).to_dict()
names_entity=j.dropna(subset=['entity_name']).drop_duplicates('entity_id').set_index('entity_id').entity_name.to_dict()
names_supplier=j.dropna(subset=['supplier_name']).drop_duplicates('supplier_id').set_index('supplier_id').supplier_name.to_dict()
case_rows=[]
for label, entity in choices:
    print(label, entity, names_entity.get(entity,''), 'categorías:',cat_entity.get(entity,''))
    positives=set(future.loc[(future.entity_id==entity)&future.eligible_positive,'supplier_id'])
    for method, r in rankings.items():
        selected=r.loc[r.entity_id.eq(entity)&r['rank'].le(3)].sort_values('rank')
        if selected.empty:
            case_rows.append({'caso':label,'método':method,'posición':None,'proveedor':None,'nombre':None,'categorías proveedor':None,'positivo 2023':None})
        for row in selected.itertuples(index=False):
            case_rows.append({'caso':label,'método':method,'posición':row.rank,'proveedor':row.supplier_id,
                              'nombre':names_supplier.get(row.supplier_id,''),'categorías proveedor':cat_supplier.get(row.supplier_id,''),
                              'positivo 2023':row.supplier_id in positives})
display(pd.DataFrame(case_rows))

ALS con acierto PE-CONSUCODE-10 JURADO NACIONAL DE ELECCIONES categorías: goods, services
ALS sin acierto PE-CONSUCODE-1000 MUNICIPALIDAD DISTRITAL DE MOLINOS - PACHITEA categorías: goods, services, works
Vecindario vacío PE-CONSUCODE-1011 MUNICIPALIDAD DISTRITAL DE RONDOS categorías: services, works


,caso,método,posición,proveedor,nombre,categorías proveedor,positivo 2023
0,ALS con acierto,popularidad,1.0,PE-RUC-20100017491,TELEFONICA DEL PERU SAA,"goods, services, works",True
1,ALS con acierto,popularidad,2.0,PE-RUC-20511037001,GRUPO SANTA FE SOCIEDAD ANONIMA CERRADA - GRUP...,goods,False
2,ALS con acierto,popularidad,3.0,PE-RUC-20501887286,DIAGNOSTICA PERUANA S.A.C.,goods,False
3,ALS con acierto,vecindario,1.0,PE-RUC-20100049008,COMPAÑIA PERUANA DE RADIODIFUSION S.A.,services,False
4,ALS con acierto,vecindario,2.0,PE-RUC-20113367360,RADIO LA KARIBEÑA SOCIEDAD ANONIMA CERRADA,services,False
5,ALS con acierto,vecindario,3.0,PE-RUC-20143229816,EMPRESA EDITORA EL COMERCIO S.A.,services,False
6,ALS con acierto,ALS,1.0,PE-RUC-20100017491,TELEFONICA DEL PERU SAA,"goods, services, works",True
7,ALS con acierto,ALS,2.0,PE-RUC-20552504641,OPTICAL TECHNOLOGIES SAC,"services, works",False
8,ALS con acierto,ALS,3.0,PE-RUC-20100210909,LA POSITIVA SEGUROS Y REASEGUROS,services,False
9,ALS sin acierto,popularidad,1.0,PE-RUC-20467534026,AMERICA MOVIL PERU S.A.C.,"goods, services, works",False


## Decisión inicial

Las tres categorías generales aportan una señal de contexto pero son demasiado amplias para demostrar sustitución. Si la cobertura de descripciones y recursos son suficientes, se puede construir un prototipo TF-IDF con perfiles de entidades y proveedores aprendidos solo de ítems anteriores al corte, y compararlo en validación 2023 con los mismos positivos y candidatos. La evaluación de 2024 ya fue observada en esta conversación y no debe presentarse como una nueva prueba final independiente. Ninguna similitud textual establece aptitud ni equivalencia de contratación.

## Resultado del prototipo textual

Configuración fijada antes de ejecutar: TF-IDF sobre descripciones de ítems de entrenamiento, perfiles sumados y normalizados por entidad y proveedor y ranking por coseno, sin fallback. Los cuatro métodos se comparan en validación 2023; la cobertura de proveedores se mide a 20. El resultado es exploratorio.

In [4]:
import numpy as np
content=path/'recommendation_content'
assert (content/'_SUCCESS').is_file()
cm=json.loads((content/'_metadata.json').read_text(encoding='utf-8'))
assert cm['input_sha256']==directory_fingerprint(base/'recommendation')
assert cm['items_sha256']==directory_fingerprint(base/'fact_award_item')
assert cm['events_sha256']==directory_fingerprint(base/'fact_award_supplier')
summary=[]
for label,folder in [('popularidad','baseline'),('vecindario','neighbors'),('ALS','als'),('contenido','content')]:
    m=json.loads((path/f'recommendation_{folder}/_metadata.json').read_text(encoding='utf-8'))
    if label=='ALS':
        result=next(r for r in m['results'] if r['config']==m['selected'])
        scores=result['metrics_macro']; cover=result['provider_coverage']; n=m['entities_evaluated']
    else:
        scores=m['metrics_macro']; cover=m['provider_coverage']; n=m.get('entities_with_ranking',m['entities_evaluated'])
    summary.append({'método':label,'recall@10':scores['10']['recall'],'ndcg@10':scores['10']['ndcg'],
                    'recall@20':scores['20']['recall'],'ndcg@20':scores['20']['ndcg'],
                    'entidades con ranking':n,'cobertura proveedores @20':cover})
display(pd.DataFrame(summary).set_index('método').round(4))
print('Diagnóstico de contenido:',cm['diagnostics'])

,recall@10,ndcg@10,recall@20,ndcg@20,entidades con ranking,cobertura proveedores @20
método,,,,,,
popularidad,0.0257,0.0217,0.0342,0.0238,1843,0.0012
vecindario,0.0232,0.0282,0.0335,0.0297,1607,0.3353
ALS,0.0555,0.0558,0.0760,0.0607,1843,0.0436
contenido,0.0109,0.0098,0.0210,0.0126,1842,0.4421


Diagnóstico de contenido: {'documents': 50647, 'tokens_before_vectorization': 834719, 'terms': 30000, 'document_nonzero': 1184429, 'estimated_sparse_bytes': 100541648, 'entities_with_profile': 2687, 'suppliers_with_profile': 28818}


### Diferencias emparejadas por entidad

Remuestreo bootstrap de 1 000 réplicas con semilla 42 sobre entidades comunes y NDCG@10. La incertidumbre no corrige la cobertura incompleta, la posible dependencia entre entidades, las versiones de los datos ni la selección del experimento.

In [5]:
frames={}
for label,folder in [('popularidad','baseline'),('vecindario','neighbors'),('ALS','als'),('contenido','content')]:
    x=pd.read_parquet(path/f'recommendation_{folder}/metrics_by_entity.parquet')
    x=x[x.k==10]
    if label=='ALS': x=x[x['rank']==json.loads((path/'recommendation_als/_metadata.json').read_text(encoding='utf-8'))['selected']['rank']]
    assert not x.entity_id.duplicated().any()
    frames[label]=x.set_index('entity_id').ndcg
wide=pd.concat(frames,axis=1).sort_index(); assert wide.notna().all().all() and len(wide)==cm['entities_evaluated']
rng=np.random.default_rng(42); idx=rng.integers(0,len(wide),size=(1000,len(wide)))
comparisons=[]
for other in ('popularidad','vecindario','ALS'):
    delta=(wide['contenido']-wide[other]).to_numpy(); samples=delta[idx].mean(axis=1)
    comparisons.append({'comparación':f'contenido − {other}','media':delta.mean(),
                        'IC 2,5 %':np.quantile(samples,.025),'IC 97,5 %':np.quantile(samples,.975)})
display(pd.DataFrame(comparisons).set_index('comparación').round(4))

,media,"IC 2,5 %","IC 97,5 %"
comparación,,,
contenido − popularidad,-0.0119,-0.0152,-0.0082
contenido − vecindario,-0.0185,-0.0232,-0.0137
contenido − ALS,-0.0460,-0.0520,-0.0401


### Ejemplos de texto de entrenamiento

Se muestran descripciones concretas usadas para el perfil de cada entidad y del proveedor situado primero en su ranking de contenido. Son ejemplos de sus antecedentes, no necesariamente los términos de mayor peso ni una explicación completa de la puntuación. La coincidencia textual no demuestra elegibilidad o sustitución.

In [6]:
r=pd.read_parquet(content/'rankings.parquet')
samples=[]
for label, entity in choices:
    ranked=r[(r.entity_id==entity)&(r['rank']==1)]
    if ranked.empty: continue
    supplier=ranked.iloc[0].supplier_id
    entity_text=sorted(set(items.loc[items.entity_id.eq(entity),'item_description'].dropna().astype(str)))[:2]
    supplier_text=sorted(set(items.loc[items.supplier_id.eq(supplier),'item_description'].dropna().astype(str)))[:2]
    pos=set(future.loc[(future.entity_id==entity)&future.eligible_positive,'supplier_id'])
    samples.append({'caso':label,'entidad':entity,'proveedor':supplier,
                    'textos entidad':entity_text,'textos proveedor':supplier_text,
                    'positivo observado 2023':supplier in pos})
for row in samples:
    print('\n',row['caso'], row['entidad'],'→',row['proveedor'],'positivo:',row['positivo observado 2023'])
    print('  entidad:',row['textos entidad']); print('  proveedor:',row['textos proveedor'])


 ALS con acierto PE-CONSUCODE-10 → PE-RUC-10316248522 positivo: False
  entidad: ['ADQUISICION DE 04 ESCRITORIOS AGLOMERADO ENCHAPADO EN MADERA - ELECCIONES MUNICIPALES 2015', 'ADQUISICION DE ESTANTES DE METAL PARA LA IMPLEMENTACION DEL ARCHIVO CENTRAL DE LA SEDE CUSCO']
  proveedor: ['SERVICIO DE ALQUILER DE INMUEBLE PARA LA OFICINA DESCONCENTRADA DE ANCASH']

 ALS sin acierto PE-CONSUCODE-1000 → PE-RUC-10224770273 positivo: False
  entidad: ['CONT P LA FOR DL ESTU D PRE INV A NIV D PERF Y ELA DE EX TEC DL PROY REHA Y MEJ DL CV CAJ YURA  I CALLA MANZ RIN PALMA II CHINCH PALMA JARA PERGA RANG  III COCHA COCHA ALTO IV ACHI ANCO V PUCA CUBA VI ORO HICHA USHA VII MOL PACH HCO', 'CONTRATACION DEL SERVICIO DE CONSULTORIA DE OBRA PARA LA ELABORACION DEL EXPEDIENTE TECNICO DEL PROYECTO INSTALACION DEL SISTEMA DE RIEGO TECNIFICADO DEL CASERIO DE COCHATO-MOLINO-PACHITEA-HUANUCO']
  proveedor: ['ELABORACION DE UN PERFIL TECNICO CONSTRUCCION DE TROCHA CARROZABLE TAMBOPATA CRUZ PUNTA CARAN DISTRI